# **Exercise 1 – Kenya: Loading Historical Sensor Data into HydroServer**

### **Overview**

This exercise guides you through creating a HydroServer workspace, a monitoring site, and adatastream, and then loading stage observations from the Mara Hydrological Monitoring Station (81269).

You can access additional information about the station through the [Rwanda Water Data Portal](https://waterportal.rwb.rw/index.php/location_ng_info/259501)

### **Description**

This Jupyter Notebook demonstrates how to load time series data for a monitoring site from a comma separate values (CSV) file into a datastream in HydroServer. It demonstrates the following:

1. Connects to HydroServer
2. Creates a workspace
3. Creates a monitoring site (Thing)
4. Creates observation method metadata
5. Creates observed property metadata
6. Creates a unit for the observed property
7. Creates a processing level
8. Creates a datastream for the observed property
9. Loads observations from a CSV file into the datastream

More detailed examples of how to use hydroserverpy are available in HydroServer's documentation at:

* https://hydroserver2.github.io/hydroserver/user-guides/how-to/using-the-python-client.html
* https://www.hydroserver.org

### **Prerequisites**

You must have an account on the HydroServer Playground instance to run this notebook. If you haven't set up your user account yet, navigate to https://playground.hydroserver.org and follow the instructions to create a new user account.

**Note: If your Google Colab session disconnects, reconnect to the runtime before continuing. Resources that you have already created in HydroServer will not be lost. Avoid rerunning cells that create resources, as this may result in duplicate resources or errors.**

### **References**

Code adapted by Sara Alonso Vicario from the Center for Geospatial Solutions from examples developed by the HydroServer development team, including Jeff Horsburgh, Ken Lippold, and Daniel Slaugh. The original materials are available in this [HydroShare resource](https://www.hydroshare.org/resource/136c2bc6512540d59ce707bbd9c93e8a/).


**Note: If your Google Colab session disconnects, reconnect to the runtime before continuing. Resources that you have already created in HydroServer will not be lost. Avoid rerunning cells that create resources, as this may result in duplicate resources or errors.**

### **References**

Code adapted by Sara Alonso Vicario from the Center for Geospatial Solutions from examples developed by the HydroServer development team, including Jeff Horsburgh, Ken Lippold, and Daniel Slaugh. The original materials are available in this [HydroShare resource](https://www.hydroshare.org/resource/136c2bc6512540d59ce707bbd9c93e8a/).

## 1. **Getting Started**

---

### **Install hydroserverpy**

For this workshop, we will use Google Colab to run the exercises. Before starting, run the code cell below to install the required version of the hydroserverpy package. The current version of hydroserverpy used for this training is [1.11.3.](https://pypi.org/project/hydroserverpy/)

In [1]:
!pip install hydroserverpy==1.11.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 5.2 MB/s eta 0:00:00


### **Import Required Packages**

The following packages and modules are used in this exercise:

- **hydroserverpy** – Connects to HydroServer and allows us to create and manage HydroServer resources programmatically.
- **pandas** – Reads, organizes, and processes historical sensor data.
- **datetime** – Works with dates and times.
- **getpass** – Allows you to enter your HydroServer password securely without displaying it on the screen.

In [2]:
# Import HydroServer to connect to and manage HydroServer resources
from hydroserverpy import HydroServer

# Import pandas to read, organize, and process the sensor data
import pandas as pd

# Import datetime to work with dates and times
from datetime import datetime

# Import getpass to securely enter your HydroServer password
from getpass import getpass

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to establish a connection to it. For this example, we will use your username (email) and password because we will create the workspace programmatically.

When you run the code, you will be prompted to enter your password.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [3]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

Enter your HydroServer password: ··········


### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [4]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


## 2. **Create a Workspace**

---

Now that we are connected to HydroServer, we can create a **Workspace** to store your monitoring sites and data.

A Workspace is an organizational container where you can create monitoring sites and datastreams, and load observations.

Access control is also applied at the Workspace level in HydroServer. You can give read-only or edit permission to other HydroServer users to work in your Workspace.

**IMPORTANT: In the following cell, change the name of the workspace using your name so it is unique to you.**

In [5]:
# Change the workspace name here using your first name so it is unique to you
workspace_name = "Kenya Training 2026 - your name"  # Choose a unique workspace name
new_workspace = hs.workspaces.create(name=workspace_name, is_private=False)

# Get the UUID of the workspace you just created - we'll use it later
workspace_id = new_workspace.uid

# Print some properties of the workspace
print(f"Created workspace named {new_workspace.name}")
print(f"Workspace ID: {workspace_id}")

Created workspace named Kenya Training 2026 - your name
Workspace ID: 01a07c47-90d4-73c3-911b-d51d0311b2fb


## 3. **Create the Monitoring Site**

---

We are going to load historical river stage data from the Mara Hydrological Station.

First, we need to create a monitoring site, also referred to as a **Thing** in HydroServer. HydroServer uses a modified version of the [OGC SensorThings API data model](https://www.ogc.org/standards/sensorthings/), where a **Thing** represents a monitoring location where observations are collected.

Once the monitoring site is created, HydroServer automatically assigns it a Universally Unique Identifier (UUID). We can use this UUID to build the URL for the site's landing page in HydroServer.

In [6]:
# Create a Thing for the Kanzenze Hydrological Station

new_thing = hs.things.create(
    workspace=workspace_id,
    name='Mara at Mara Rianda Hydrological Station',
    description='NBI Hydromet monitoring station on the Mara River at Mara Rianda, Kenya.',
    sampling_feature_type='Site',
    sampling_feature_code='Mara_Rianda',
    site_type='Stream',
    #latitude, longitude, and elevation are required to save the monitoring site
    latitude=-1.184, #approximate need to be updated
    longitude=35.119, #approximate need to be updated
    elevation_m=1690.0, #approximate need to be updated
    admin_area_1='Narok County',
    country='KE',
    data_disclaimer='Data provided through the Nile Basin Initiative (NBI) Hydromet system.',
    is_private=False
)

# Get the ID for the new Thing and print its HydroServer landing page

thing_id = new_thing.uid

print(f'Created new thing with ID: {thing_id}')
print('You can access the new Thing in the HydroServer Data Management App at:')
print(f'{hydroserver_host}/sites/{thing_id}')

Created new thing with ID: 01a07c47-9f8d-7107-8998-9a4c8733a669
You can access the new Thing in the HydroServer Data Management App at:
https://playground.hydroserver.org/sites/01a07c47-9f8d-7107-8998-9a4c8733a669


## 4. **Create the DataStream**

---

In the following sections, we will create the necessary metadata to load data for a time series of observations recorded at a monitoring site. You can create this metadata using the web user interface of the Data Management App, or you can do it using code, which we are demonstrating here.

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Observation Method**: The instrument or method used to measure or create the Observation values.
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create an Observation Method (Sensor)**

The OGC SensorThings API data model refers to the method used for creating observations as the "Sensor". In many cases this will be a physical sensor installed at the monitoring site. But, sometimes other methods are used to create observations. We need to create the metadata describing this so potential data users know how the data were created.

**NOTE**: The specific metadata required when creating metadata for a Sensor is dependent upon the "Method Type". For instrument deployments, specific information about the manufacturer and model of the sensor should be specified. For "Derivation" methods, the name and description are required, and a method_code and method_link can be specified if needed.

In [7]:
mara_stage_sensor = hs.sensors.create(
    workspace=workspace_id,
    name='Mara Rianda River Stage Observations',
    description='River stage observations recorded at the Mara at Mara Rianda hydrological monitoring station in Kenya.',
    encoding_type='application/json',
    method_type='Observation',
    method_code='mara-rianda-stage'
)

### Create an Observed Property

An **Observed Property** defines the variable being measured at a monitoring site. As with the monitoring site (Thing), we need to provide the required and optional metadata that describe the Observed Property.

In this example, we will create an Observed Property for **river stage**, since the CSV file we will upload contains historical river stage observations.

In [8]:
stage = hs.observedproperties.create(
    workspace=workspace_id,
    name='Stage',
    definition='Stage',
    description='Stage is the height of the water surface at a monitoring location relative to a reference level.',
    observed_property_type='Hydrology',
    code='Stage'
)

print("Created observed property:")
print(f"{stage.name}: {stage.uid}")

Created observed property:
Stage: 01a07c47-be2f-7e19-abf8-9239da25ec39


### **Create Units of Measure**

Next, we need to add metadata specifying the unit of measurement used for the observations in the CSV file.

In [9]:
stage_unit = hs.units.create(
    workspace=workspace_id,
    name='Meter',
    symbol='m',
    definition='Unit for water stage',
    unit_type='Length'
)

print("Created unit:")
print(f"{stage_unit.name}: {stage_unit.uid}")

Created unit:
Meter: 01a07c47-c828-7ba2-9059-d6a1be46b3ee


### **Create a Processing Level**

In HydroServer, the Processing Level indicates the degree of processing an observation has been subject to. F

or example, data can be "Raw", which means that they were recorded in the field and nobody has looked at them yet, or they could be "Quality Controlled", which means that a technician has reviewed the data.

We are considering these observations to be raw data with no additional processing, so we need to define a Processing Level that indicates this.

In [10]:
new_processing_level = hs.processinglevels.create(
    workspace=workspace_id,
    code='Raw',
    definition='Raw Data',
    explanation='Data that have not been processed or quality controlled.'
)

print("Created processing levels:")
print(f"{new_processing_level.code}: {new_processing_level.uid}")

Created processing levels:
Raw: 01a07c47-cfb2-75e3-8183-559d3e40fb8c


### **Configure the Datastream**

The final step before loading the observations is to create a Datastream. A datastream describes the time series and connects it to the relevant metadata, including where the observations were collected, what variable was measured, which observation method was used, the unit of measurement, and the processing level.

In the following code, we create the datastream by linking the metadata resources created in the previous steps using their unique identifiers (UIDs). We also define additional datastream-specific metadata, such as the data characteristics, time spacing, name, and description.

Once the datastream is created, we can load the observations from the CSV file into it.

**NOTE**: Since these Datastreams are new, they don't contain any Observation values yet. We'll set the ```value_count=0``` and arbitrarily set the ```phenomenon_begin_time``` and ```phenomenon_end_time```. Those will get reset when we load Observation values.


In [11]:
ds_stage = hs.datastreams.create(
    name=f"{stage.name} - {new_thing.name}",
    description=f'{stage.name} observations at {new_thing.name}',
    thing=new_thing.uid,
    sensor=mara_stage_sensor.uid,
    observed_property=stage.uid,
    processing_level=new_processing_level.uid,
    unit=stage_unit.uid,
    observation_type='Field Observation',
    result_type='Timeseries',
    sampled_medium='Surface Water',
    no_data_value=-9999,
    aggregation_statistic='Continuous',
    time_aggregation_interval=0,
    time_aggregation_interval_unit='minutes',
    # The dataset predominantly contains hourly observations.
    # Some timestamps are missing, so there are gaps longer than 1 hour.
    intended_time_spacing=1,
    intended_time_spacing_unit='hours',
    # Use Ongoing because this is a current NBI Hydromet monitoring station.
    status='Ongoing',
    value_count=0,
    # First observation in the provided dataset.
    phenomenon_begin_time=datetime(year=2026, month=1, day=1),
    # Leave the end time open because monitoring is ongoing.
    phenomenon_end_time=None,
    is_private=False,
    is_visible=True
)

print("Created datastream:")
print(f"{ds_stage.name}: {ds_stage.uid}")

Created datastream:
Stage - Mara at Mara Rianda Hydrological Station: 01a07c47-d725-7bcf-80dc-7f161c3ba81b
Created datastream:
Stage - Mara at Mara Rianda Hydrological Station: 01a07c47-d725-7bcf-80dc-7f161c3ba81b


## 5. Load Time Series Data from the CSV File

---

Now that we have all of the metadata we need, we can read the CSV data file and load data. The following sections break this down using a convenience data structure for mapping the names of the columns in the CSV file to the datastream and then chunking up the CSV data to load it into HydroServer.

### **Specify the CSV File to Load Data From**

For this example, we are going to load data to a monitoring site we create in HydroServer from a comma-separated values (CSV) file. This CSV file contains stage raw data for a monitoring station. To limit the size of the requests we are making to the HydroServer API, we will also specify a chunk size for loading data. We'll load the whole file, but in requests that are sized by the chunk size we set here.

In [12]:
file_to_load = (
    "https://raw.githubusercontent.com/"
    "savicario/addis-ababa-hydroserver-training-2026/"
    "main/kenya/Exercise1/data/Mara%20River.xlsx"
)

chunk_size = 10000

print(file_to_load)

https://raw.githubusercontent.com/savicario/addis-ababa-hydroserver-training-2026/main/kenya/Exercise1/data/Mara%20River.xlsx


### **Create Datastream UUID Mapping to CSV File Columns**

 We'll first create a convenience data structure (a Python List object) to match up the column names in the CSV file with the UUID of the Datastreams.

 We'll use this to specify which columns from the file we want to load.

 Each element in the Python list will be a Tuple specifying the name of the column in the CSV file and the UUID for the datastream the values in that column should be loaded into.

**NOTE**: The datastream metadata must exist in HydroServer before loading data into it.

In [14]:
#
datastreams = [
    ('Value', ds_stage.uid)
]

print(f"Number of datastreams to load: {len(datastreams)}")

Number of datastreams to load: 1


### Read the CSV File Using Pandas

Read the CSV file into a Pandas DataFrame. Skip the metadata rows at the beginning of the file and use the next row as the column header.

The `Timestamp` column is parsed as datetime values so that the observations can be properly processed and uploaded to HydroServer.

In [15]:
# Read the Excel data file containing the observations to load.

df = pd.read_excel(
    file_to_load,
    sheet_name='in',

    # Skip the metadata rows at the beginning of the Kenya Mara River file.
    skiprows=4,

    # Use the first row after the metadata as the column header.
    header=0,

    # Parse the monitoring date and time column as datetime values.
    parse_dates=['Monitoring Date And Time']
)

print('Data file read successfully.')
print(f"Number of rows to be loaded: {len(df)}")

Data file read successfully.
Number of rows to be loaded: 3175


### **Specify the Time Zone for the Data**

If no timezone offset information is provided when loading data, HydroServer will assume the timestamps you provide are in UTC.

The data file we are loading data from already has a DateTimeUTC column, but the fact that it is UTC is not encoded in the file. So, we need to tell Pandas that those timestamps are UTC before loading the data.

In [16]:
# Convert the Kanzenze timestamps from their recorded timezone (+02:00) to UTC.
df['Monitoring Date And Time'] = pd.to_datetime(df['Monitoring Date And Time'], utc=True)

print('Monitoring Date And Time in the "Monitoring Date And Time" column have been converted to UTC.')

Monitoring Date And Time in the "Monitoring Date And Time" column have been converted to UTC.


### **Chunk the Data and Load into HydroServer**

The last step in loading data is to divide it up into reasonably sized chunks to load it into HydroServer. We set the chunk size (the number of records to add at one time) at the top of this notebook. The code below just divides the Pandas dataframe up into chunks for each datastream according to the chunk size and then loads the data one chunk at a time using the hydroserverpy ```load_observations()``` function.

In [17]:
# Select the timestamp and river stage columns.
observations = df[['Monitoring Date And Time', 'River Stage']]

# Rename the columns to the names expected by hydroserverpy.
observations = observations.rename(
    columns={
        'Monitoring Date And Time': 'phenomenon_time',
        'River Stage': 'result'
    }
)

# Get the Mara River Historical Stage datastream.
datastream = hs.datastreams.get(uid=ds_stage.uid)

# Upload the observations to HydroServer.
datastream.load_observations(observations)

print(f"Loaded {len(observations)} stage observations.")
print(f"Access the data in HydroServer at: {hydroserver_host}/sites/{thing_id}")

Loaded 3175 stage observations.
Access the data in HydroServer at: https://playground.hydroserver.org/sites/01a07c47-9f8d-7107-8998-9a4c8733a669


### What You Have Learned

You now know how to use `hydroserverpy` to:

- Connect to a HydroServer instance.
- Create a workspace.
- Create a monitoring site (Thing).
- Define the required metadata, including the observation method, observed property, unit, and processing level.
- Create a datastream and link it to the corresponding metadata.
- Prepare observations from a CSV file for upload.
- Load observations into a HydroServer datastream.
- View and manage the resulting data through the HydroServer Web Dashboard.